# Endometriosis single-cell RNA-seq analysis

This notebook develops an analysis of the processed 10x Genomics single-cell RNA-seq data associated with [Marečková *et al.*, *Nature Genetics* (2024)](https://www.nature.com/articles/s41588-024-01873-w).

The planned workflow is organized into four high-level sections:

1. **Read and organize the count matrices** (current section)
2. Quality control and normalization
3. Cell-type annotation
4. Endometriosis-focused exploration of a selected cell population

> This first section only constructs sample-level `AnnData` objects from the original filtered count matrices. It intentionally does not filter, normalize, log-transform, integrate, or annotate cells.

## 1. Read and organize the count matrices

### Input format and relationship to the paper

Each `.tar.gz` archive in `data/` represents one donor–library combination and contains a Cell Ranger-style filtered feature-barcode matrix:

- `matrix.mtx.gz`: sparse integer UMI counts, stored as genes × cells on disk
- `features.tsv.gz`: Ensembl gene IDs, gene symbols, and feature types
- `barcodes.tsv.gz`: 10x cell barcodes

The paper aligned reads to **GRCh38-2020-A** and generated filtered count matrices with **Cell Ranger 6.0.2**. `scanpy.read_10x_mtx` reads this format and returns an `AnnData` object oriented as cells × genes. We use Ensembl IDs as the gene index because they are stable and unique, while retaining gene symbols in `adata.var`.

The archive name follows `<library_id>_<donor_id>`. A donor can occur in multiple libraries, so both identifiers are retained separately. The raw 10x barcode is also retained, but the `AnnData` cell index is prefixed with the complete sample ID to prevent barcode collisions across samples.

### 1.1 Imports and project paths

The path-discovery code allows the notebook to be launched either from the project root or from the `notebooks/` directory. This notebook uses the project Conda environment defined in `environment.yml`. In JupyterLab or another IDE, select the **Endometriosis study** kernel before running the notebook.

In [12]:
from pathlib import Path
import shutil
import tarfile
import tempfile

import anndata as ad
import pandas as pd
import scanpy as sc
from scipy import sparse

sc.settings.verbosity = 2


def find_project_root(start: Path) -> Path:
    """Find the nearest parent directory containing the project's data folder."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate a parent directory containing data/.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

Project root: /Users/ale/endometriosis_study
Data directory: /Users/ale/endometriosis_study/data


### 1.2 Discover the sample archives

We enumerate archives rather than relying on a manually maintained sample list. Sorting the paths makes sample order reproducible. The table below provides a quick check of the discovered sample IDs and compressed file sizes before loading any matrices.

In [13]:
archive_paths = sorted(DATA_DIR.glob("*.tar.gz"))

if not archive_paths:
    raise FileNotFoundError(f"No .tar.gz sample archives were found in {DATA_DIR}")

archive_manifest = pd.DataFrame(
    {
        "sample_id": [path.name.removesuffix(".tar.gz") for path in archive_paths],
        "archive": [path.name for path in archive_paths],
        "size_mb": [path.stat().st_size / 1024**2 for path in archive_paths],
    }
)

print(f"Discovered {len(archive_paths)} sample archives.")
display(archive_manifest)

Discovered 35 sample archives.


,sample_id,archive,size_mb
0,FRZFRESH_GX25_ES345,FRZFRESH_GX25_ES345.tar.gz,25.058156
1,FRZFRESH_GX26_ES345,FRZFRESH_GX26_ES345.tar.gz,28.703766
2,FRZFRESH_GX27_ES345,FRZFRESH_GX27_ES345.tar.gz,31.588312
3,FRZFRESH_GX28_ES345,FRZFRESH_GX28_ES345.tar.gz,22.442861
4,HCA_A_RepT_RNA13247830_A70,HCA_A_RepT_RNA13247830_A70.tar.gz,58.950799
5,UA_Endo10298210_FX1125,UA_Endo10298210_FX1125.tar.gz,55.641235
6,UA_Endo10298210_FX1176,UA_Endo10298210_FX1176.tar.gz,29.113951
7,UA_Endo10298211_FX1125,UA_Endo10298211_FX1125.tar.gz,1.106306
8,UA_Endo10298211_FX1176,UA_Endo10298211_FX1176.tar.gz,17.975212
9,UA_Endo10298212_FX1156,UA_Endo10298212_FX1156.tar.gz,26.587353


### 1.3 Define a loader for one sample

`scanpy.read_10x_mtx` expects the three matrix files to be in a directory. The loader therefore copies only those required files from an archive into a temporary directory, reads them, and removes the temporary files automatically. The source archives remain unchanged.

For each sample, the loader:

1. validates that all three expected 10x files are present;
2. reads only `Gene Expression` features into a sparse `AnnData`;
3. records the original barcode, sample ID, library ID, and donor ID in `adata.obs`;
4. creates globally unique cell IDs of the form `<sample_id>:<barcode>`; and
5. records source information in `adata.uns`.

At this stage, `adata.X` remains the original unnormalized UMI count matrix.

In [14]:
REQUIRED_10X_FILES = ("matrix.mtx.gz", "features.tsv.gz", "barcodes.tsv.gz")


def split_sample_id(sample_id: str) -> tuple[str, str]:
    """Split '<library_id>_<donor_id>' at its final underscore."""
    try:
        library_id, donor_id = sample_id.rsplit("_", maxsplit=1)
    except ValueError as error:
        raise ValueError(
            f"Sample ID {sample_id!r} does not match '<library_id>_<donor_id>'."
        ) from error
    return library_id, donor_id


def read_sample_archive(archive_path: Path) -> ad.AnnData:
    """Read one archived 10x matrix and attach sample provenance."""
    sample_id = archive_path.name.removesuffix(".tar.gz")
    library_id, donor_id = split_sample_id(sample_id)
    archive_prefix = f"work/{sample_id}"

    with tempfile.TemporaryDirectory(prefix=f"{sample_id}_") as temporary_dir:
        matrix_dir = Path(temporary_dir)

        with tarfile.open(archive_path, mode="r:gz") as archive:
            for filename in REQUIRED_10X_FILES:
                member_name = f"{archive_prefix}/{filename}"
                try:
                    member = archive.getmember(member_name)
                except KeyError as error:
                    raise FileNotFoundError(
                        f"{archive_path.name} is missing {member_name}."
                    ) from error

                source = archive.extractfile(member)
                if source is None:
                    raise OSError(f"Could not read {member_name} from {archive_path.name}.")

                with source, (matrix_dir / filename).open("wb") as destination:
                    shutil.copyfileobj(source, destination)

        adata = sc.read_10x_mtx(
            matrix_dir,
            var_names="gene_ids",
            make_unique=True,
            gex_only=True,
            cache=False,
        )

    # CSR is efficient for the later cell-wise QC calculations.
    if not sparse.isspmatrix_csr(adata.X):
        adata.X = sparse.csr_matrix(adata.X)

    original_barcodes = adata.obs_names.astype(str)
    adata.obs["barcode"] = original_barcodes
    adata.obs["sample_id"] = sample_id
    adata.obs["library_id"] = library_id
    adata.obs["donor_id"] = donor_id
    adata.obs_names = pd.Index(
        [f"{sample_id}:{barcode}" for barcode in original_barcodes],
        name="cell_id",
    )
    adata.var_names.name = "gene_id"

    if not adata.obs_names.is_unique:
        raise ValueError(f"Cell IDs are not unique within {sample_id}.")

    adata.uns["sample_id"] = sample_id
    adata.uns["library_id"] = library_id
    adata.uns["donor_id"] = donor_id
    adata.uns["source_archive"] = str(archive_path.relative_to(PROJECT_ROOT))
    adata.uns["matrix_description"] = (
        "Cell Ranger filtered, unnormalized gene-expression UMI counts"
    )

    return adata

### 1.4 Load all samples as `AnnData` objects

The objects are stored in a dictionary keyed by sample ID. Keeping them separate is useful for sample-level inspection and QC; concatenation will be handled deliberately in the next section. Because the matrices are sparse, zeros are not materialized in memory, but loading all samples can still require several gigabytes of RAM.

In [15]:
sample_adatas: dict[str, ad.AnnData] = {}

for position, archive_path in enumerate(archive_paths, start=1):
    sample_id = archive_path.name.removesuffix(".tar.gz")
    print(f"[{position:>2}/{len(archive_paths)}] Reading {sample_id}")
    sample_adatas[sample_id] = read_sample_archive(archive_path)

print(f"\nLoaded {len(sample_adatas)} AnnData objects.")

[ 1/35] Reading FRZFRESH_GX25_ES345
[ 2/35] Reading FRZFRESH_GX26_ES345
[ 3/35] Reading FRZFRESH_GX27_ES345
[ 4/35] Reading FRZFRESH_GX28_ES345
[ 5/35] Reading HCA_A_RepT_RNA13247830_A70
[ 6/35] Reading UA_Endo10298210_FX1125
[ 7/35] Reading UA_Endo10298210_FX1176
[ 8/35] Reading UA_Endo10298211_FX1125
[ 9/35] Reading UA_Endo10298211_FX1176
[10/35] Reading UA_Endo10298212_FX1156
[11/35] Reading UA_Endo10298212_FX9006
[12/35] Reading UA_Endo10298213_FX1156
[13/35] Reading UA_Endo10298213_FX9006
[14/35] Reading UA_Endo12680031_FX1119
[15/35] Reading UA_Endo12680031_FX1259
[16/35] Reading UA_Endo12680032_FX1259
[17/35] Reading UA_Endo12680033_FX1249
[18/35] Reading UA_Endo12680033_FX1254
[19/35] Reading UA_Endo12680034_FX1249
[20/35] Reading UA_Endo12680034_FX1254
[21/35] Reading UA_Endo12961679_FX1285
[22/35] Reading UA_Endo12961679_SE02
[23/35] Reading UA_Endo12961680_FX1285
[24/35] Reading UA_Endo12961680_SE02
[25/35] Reading UA_Endo12961681_FX1289
[26/35] Reading UA_Endo12961681_SE03


### 1.5 Summarize the loaded objects

This manifest verifies the dimensions and sparse-matrix representation of every object. `n_cells` is the number of filtered barcodes assigned to that donor–library combination; it is not the number of independent biological replicates.

In [16]:
sample_summary = pd.DataFrame(
    [
        {
            "sample_id": sample_id,
            "library_id": adata.uns["library_id"],
            "donor_id": adata.uns["donor_id"],
            "n_cells": adata.n_obs,
            "n_genes": adata.n_vars,
            "nonzero_values": adata.X.nnz,
            "matrix_dtype": str(adata.X.dtype),
            "sparse_format": adata.X.format,
        }
        for sample_id, adata in sample_adatas.items()
    ]
).sort_values("sample_id", ignore_index=True)

display(sample_summary)
print(f"Total matrix columns/cells: {sample_summary['n_cells'].sum():,}")
print(f"Unique donors: {sample_summary['donor_id'].nunique():,}")
print(f"Unique library IDs: {sample_summary['library_id'].nunique():,}")

,sample_id,library_id,donor_id,n_cells,n_genes,nonzero_values,matrix_dtype,sparse_format
0,FRZFRESH_GX25_ES345,FRZFRESH_GX25,ES345,2849,36601,8307210,float32,csr
1,FRZFRESH_GX26_ES345,FRZFRESH_GX26,ES345,2371,36601,9621086,float32,csr
2,FRZFRESH_GX27_ES345,FRZFRESH_GX27,ES345,3507,36601,10479303,float32,csr
3,FRZFRESH_GX28_ES345,FRZFRESH_GX28,ES345,1855,36601,7476524,float32,csr
4,HCA_A_RepT_RNA13247830_A70,HCA_A_RepT_RNA13247830,A70,7067,36601,19963018,float32,csr
5,UA_Endo10298210_FX1125,UA_Endo10298210,FX1125,4593,36601,18969548,float32,csr
6,UA_Endo10298210_FX1176,UA_Endo10298210,FX1176,2535,36601,9908912,float32,csr
7,UA_Endo10298211_FX1125,UA_Endo10298211,FX1125,45,36601,269696,float32,csr
8,UA_Endo10298211_FX1176,UA_Endo10298211,FX1176,943,36601,5948980,float32,csr
9,UA_Endo10298212_FX1156,UA_Endo10298212,FX1156,3229,36601,8951641,float32,csr


Total matrix columns/cells: 94,323
Unique donors: 16
Unique library IDs: 22


### 1.6 Display the structure of one `AnnData` object

We select `UA_Endo10298211_FX1125` because it has already been inspected manually. Evaluating an `AnnData` object displays its dimensions and populated annotation slots. The following cells also preview its cell metadata (`obs`), gene metadata (`var`), sparse matrix properties, and stored provenance (`uns`).

In [17]:
example_sample_id = "UA_Endo10298211_FX1125"
if example_sample_id not in sample_adatas:
    example_sample_id = next(iter(sample_adatas))

example_adata = sample_adatas[example_sample_id]
example_adata

AnnData object with n_obs × n_vars = 45 × 36601
    obs: 'barcode', 'sample_id', 'library_id', 'donor_id'
    var: 'gene_symbols', 'feature_types'
    uns: 'sample_id', 'library_id', 'donor_id', 'source_archive', 'matrix_description'

In [18]:
print("Cell annotations (adata.obs):")
display(example_adata.obs.head())

print("Gene annotations (adata.var):")
display(example_adata.var.head())

print("Count matrix (adata.X):")
print(f"  Python type: {type(example_adata.X).__name__}")
print(f"  Shape: {example_adata.X.shape} (cells × genes)")
print(f"  Data type: {example_adata.X.dtype}")
print(f"  Nonzero values: {example_adata.X.nnz:,}")

print("Stored provenance (adata.uns):")
display(example_adata.uns)

Cell annotations (adata.obs):


,barcode,sample_id,library_id,donor_id
cell_id,,,,
UA_Endo10298211_FX1125:AAGCGAGAGCTAAATG-1,AAGCGAGAGCTAAATG-1,UA_Endo10298211_FX1125,UA_Endo10298211,FX1125
UA_Endo10298211_FX1125:ACCAAACCACACGTGC-1,ACCAAACCACACGTGC-1,UA_Endo10298211_FX1125,UA_Endo10298211,FX1125
UA_Endo10298211_FX1125:ACTACGAGTGCATGAG-1,ACTACGAGTGCATGAG-1,UA_Endo10298211_FX1125,UA_Endo10298211,FX1125
UA_Endo10298211_FX1125:ACTGTCCCATCAGCAT-1,ACTGTCCCATCAGCAT-1,UA_Endo10298211_FX1125,UA_Endo10298211,FX1125
UA_Endo10298211_FX1125:AGAGCCCTCATCACCC-1,AGAGCCCTCATCACCC-1,UA_Endo10298211_FX1125,UA_Endo10298211,FX1125


Gene annotations (adata.var):


,gene_symbols,feature_types
gene_id,,
ENSG00000243485,MIR1302-2HG,Gene Expression
ENSG00000237613,FAM138A,Gene Expression
ENSG00000186092,OR4F5,Gene Expression
ENSG00000238009,AL627309.1,Gene Expression
ENSG00000239945,AL627309.3,Gene Expression


Count matrix (adata.X):
  Python type: csr_matrix
  Shape: (45, 36601) (cells × genes)
  Data type: float32
  Nonzero values: 269,696
Stored provenance (adata.uns):


OrderedDict([('sample_id', 'UA_Endo10298211_FX1125'),
             ('library_id', 'UA_Endo10298211'),
             ('donor_id', 'FX1125'),
             ('source_archive', 'data/UA_Endo10298211_FX1125.tar.gz'),
             ('matrix_description',
              'Cell Ranger filtered, unnormalized gene-expression UMI counts')])

### 1.7 Add curated donor and library metadata

The count-matrix archives contain identifiers but not the biological covariates needed for QC stratification or endometriosis comparisons. The project-level manifest in `metadata/sample_metadata.csv` reconciles three authoritative sources:

- [ArrayExpress E-MTAB-14039](https://www.ebi.ac.uk/biostudies/arrayexpress/studies/E-MTAB-14039): donor–library pairing, disease, menstrual-cycle stage, and 10x assay;
- Supplementary Table 1 of [Marečková *et al.*](https://doi.org/10.1038/s41588-024-01873-w): donor age, tissue location, hormone exposure, endometriosis stage and assignment, other pathology, and tissue processing; and
- Supplementary Table 2: Cell Ranger summaries for each unsplit sequencing library.

The manifest has one row per local donor–library archive, keyed by `sample_id`. Metadata levels remain explicit: `donor_id` identifies the biological replicate, `library_id` identifies the technical sequencing unit, and `sample_id` identifies a donor-specific matrix after demultiplexing. Cell Ranger library metrics can therefore repeat across donor-specific matrices from the same multiplexed library and must not be interpreted as per-cell values or independent biological observations.

Analysis-relevant donor and sample covariates are broadcast into `adata.obs`, making them available for cell-level grouping and plotting. The complete record, including technical library metrics and source identifiers, is stored once in `adata.uns['sample_metadata']` to avoid unnecessarily repeating long provenance fields for every cell.

#### 1.7.1 Read and validate the metadata manifest

Before attaching metadata, we require a one-to-one match between manifest rows and loaded `AnnData` objects. These checks prevent silent sample omission, duplicate metadata rows, or mismatched donor/library identifiers.

In [19]:
METADATA_PATH = PROJECT_ROOT / "metadata" / "sample_metadata.csv"

sample_metadata = pd.read_csv(
    METADATA_PATH,
    dtype={
        "sample_id": "string",
        "archive_filename": "string",
        "library_id": "string",
        "donor_id": "string",
        "endometrial_pathology_code": "string",
    },
).sort_values("sample_id", ignore_index=True)

if not sample_metadata["sample_id"].is_unique:
    duplicates = sample_metadata.loc[
        sample_metadata["sample_id"].duplicated(keep=False), "sample_id"
    ].tolist()
    raise ValueError(f"Duplicate sample IDs in metadata: {duplicates}")

loaded_ids = set(sample_adatas)
metadata_ids = set(sample_metadata["sample_id"])
missing_metadata = sorted(loaded_ids - metadata_ids)
unexpected_metadata = sorted(metadata_ids - loaded_ids)

if missing_metadata or unexpected_metadata:
    raise ValueError(
        "Metadata/sample mismatch. "
        f"Missing metadata: {missing_metadata}; "
        f"unexpected metadata: {unexpected_metadata}."
    )

metadata_by_sample = sample_metadata.set_index("sample_id", verify_integrity=True)

print(f"Validated {len(sample_metadata)} donor–library samples.")
print(f"Biological donors: {sample_metadata['donor_id'].nunique()}")
print(f"Sequencing libraries: {sample_metadata['library_id'].nunique()}")

Validated 35 donor–library samples.
Biological donors: 16
Sequencing libraries: 22


#### 1.7.2 Inspect the biological design

The preview focuses on variables that will be important for QC plots and downstream biological comparisons. Counts of cases and controls are summarized at the donor level—not the archive or cell level—to avoid pseudoreplication from donors represented in multiple libraries.

In [20]:
METADATA_PREVIEW_COLUMNS = [
    "sample_id",
    "library_id",
    "donor_id",
    "case_control",
    "age_years",
    "cycle_context",
    "menstrual_cycle_stage_fine",
    "hormonal_treatment",
    "endometriosis_stage",
    "endometrial_pathology",
    "tissue_location",
]

display(sample_metadata[METADATA_PREVIEW_COLUMNS])

donor_metadata = sample_metadata.drop_duplicates("donor_id")
donor_design = pd.crosstab(
    donor_metadata["case_control"],
    donor_metadata["cycle_context"],
    margins=True,
)
donor_design.index.name = "case_control (unique donors)"
display(donor_design)

,sample_id,library_id,donor_id,case_control,age_years,cycle_context,menstrual_cycle_stage_fine,hormonal_treatment,endometriosis_stage,endometrial_pathology,tissue_location
0,FRZFRESH_GX25_ES345,FRZFRESH_GX25,ES345,control,40,natural_cycle,Secretory Early,none,0,none,endometrium
1,FRZFRESH_GX26_ES345,FRZFRESH_GX26,ES345,control,40,natural_cycle,Secretory Early,none,0,none,endometrium
2,FRZFRESH_GX27_ES345,FRZFRESH_GX27,ES345,control,40,natural_cycle,Secretory Early,none,0,none,endometrium
3,FRZFRESH_GX28_ES345,FRZFRESH_GX28,ES345,control,40,natural_cycle,Secretory Early,none,0,none,endometrium
4,HCA_A_RepT_RNA13247830_A70,HCA_A_RepT_RNA13247830,A70,control,37,natural_cycle,Proliferative,none,0,none,whole_uterus
5,UA_Endo10298210_FX1125,UA_Endo10298210,FX1125,case,23,natural_cycle,Proliferative,none,3,endometriosis,endometrium
6,UA_Endo10298210_FX1176,UA_Endo10298210,FX1176,case,31,natural_cycle,Proliferative,none,4,endometriosis,endometrium
7,UA_Endo10298211_FX1125,UA_Endo10298211,FX1125,case,23,natural_cycle,Proliferative,none,3,endometriosis,endometrium
8,UA_Endo10298211_FX1176,UA_Endo10298211,FX1176,case,31,natural_cycle,Proliferative,none,4,endometriosis,endometrium
9,UA_Endo10298212_FX1156,UA_Endo10298212,FX1156,case,38,natural_cycle,Secretory Mid,none,4,endometriosis;fibroids,endometrium


cycle_context,exogenous_hormones,natural_cycle,All
case_control (unique donors),,,
case,2,7,9
control,0,7,7
All,2,14,16


#### 1.7.3 Attach metadata to each `AnnData` object

Only fields that are meaningful for cell-level grouping are copied into `adata.obs`. The complete sample record is retained in `adata.uns['sample_metadata']`. Existing identifiers are checked against the manifest before any values are attached.

In [21]:
OBS_METADATA_COLUMNS = [
    "source_dataset",
    "organism",
    "tissue",
    "tissue_location",
    "assay",
    "disease",
    "case_control",
    "age_years",
    "cycle_context",
    "menstrual_cycle_stage",
    "menstrual_cycle_stage_fine",
    "hormonal_treatment",
    "endometriosis_stage",
    "endometriosis_stage_assignment",
    "endometrial_pathology_code",
    "endometrial_pathology",
    "tissue_dissociation_processing",
]

for sample_id, adata in sample_adatas.items():
    record = metadata_by_sample.loc[sample_id]

    if record["library_id"] != adata.uns["library_id"]:
        raise ValueError(f"Library mismatch for {sample_id}.")
    if record["donor_id"] != adata.uns["donor_id"]:
        raise ValueError(f"Donor mismatch for {sample_id}.")

    for column in OBS_METADATA_COLUMNS:
        adata.obs[column] = record[column]

    adata.uns["sample_metadata"] = record.to_dict()

print("Metadata attached to every AnnData object.")

Metadata attached to every AnnData object.


#### 1.7.4 Verify one annotated object and update the sample summary

The example below confirms that biological covariates are available in `obs` for individual cells, while library-level QC and source provenance are stored once in `uns`. We also extend `sample_summary` with selected design variables for use in the QC section.

In [22]:
display(example_adata.obs[["donor_id", "case_control", "menstrual_cycle_stage_fine", "endometriosis_stage"]].head())
display(pd.Series(example_adata.uns["sample_metadata"], name=example_sample_id))

SUMMARY_METADATA_COLUMNS = [
    "sample_id",
    "source_dataset",
    "case_control",
    "cycle_context",
    "menstrual_cycle_stage_fine",
    "endometriosis_stage",
    "tissue_location",
    "cellranger_called_cells",
    "cellranger_reads_per_cell",
    "cellranger_genes_per_cell",
]

sample_summary = sample_summary.drop(
    columns=[
        column
        for column in SUMMARY_METADATA_COLUMNS
        if column != "sample_id" and column in sample_summary.columns
    ]
)

sample_summary = sample_summary.merge(
    sample_metadata[SUMMARY_METADATA_COLUMNS],
    on="sample_id",
    how="left",
    validate="one_to_one",
)
display(sample_summary)

,donor_id,case_control,menstrual_cycle_stage_fine,endometriosis_stage
cell_id,,,,
UA_Endo10298211_FX1125:AAGCGAGAGCTAAATG-1,FX1125,case,Proliferative,3
UA_Endo10298211_FX1125:ACCAAACCACACGTGC-1,FX1125,case,Proliferative,3
UA_Endo10298211_FX1125:ACTACGAGTGCATGAG-1,FX1125,case,Proliferative,3
UA_Endo10298211_FX1125:ACTGTCCCATCAGCAT-1,FX1125,case,Proliferative,3
UA_Endo10298211_FX1125:AGAGCCCTCATCACCC-1,FX1125,case,Proliferative,3


archive_filename                                      UA_Endo10298211_FX1125.tar.gz
library_id                                                          UA_Endo10298211
donor_id                                                                     FX1125
source_dataset                                             Mareckova-cells & nuclei
organism                                                               Homo sapiens
tissue                                                                  endometrium
tissue_location                                                         endometrium
assay                                                                     scRNA-seq
disease                                                               endometriosis
case_control                                                                   case
age_years                                                                        23
cycle_context                                                         natura

,sample_id,library_id,donor_id,n_cells,n_genes,nonzero_values,matrix_dtype,sparse_format,source_dataset,case_control,cycle_context,menstrual_cycle_stage_fine,endometriosis_stage,tissue_location,cellranger_called_cells,cellranger_reads_per_cell,cellranger_genes_per_cell
0,FRZFRESH_GX25_ES345,FRZFRESH_GX25,ES345,2849,36601,8307210,float32,csr,Mareckova-cells,control,natural_cycle,Secretory Early,0,endometrium,6425,24482,1512
1,FRZFRESH_GX26_ES345,FRZFRESH_GX26,ES345,2371,36601,9621086,float32,csr,Mareckova-cells,control,natural_cycle,Secretory Early,0,endometrium,5403,32082,3309
2,FRZFRESH_GX27_ES345,FRZFRESH_GX27,ES345,3507,36601,10479303,float32,csr,Mareckova-cells,control,natural_cycle,Secretory Early,0,endometrium,6335,30840,2679
3,FRZFRESH_GX28_ES345,FRZFRESH_GX28,ES345,1855,36601,7476524,float32,csr,Mareckova-cells,control,natural_cycle,Secretory Early,0,endometrium,4577,35786,3332
4,HCA_A_RepT_RNA13247830_A70,HCA_A_RepT_RNA13247830,A70,7067,36601,19963018,float32,csr,Mareckova-cells,control,natural_cycle,Proliferative,0,whole_uterus,8466,103917,2426
5,UA_Endo10298210_FX1125,UA_Endo10298210,FX1125,4593,36601,18969548,float32,csr,Mareckova-cells & nuclei,case,natural_cycle,Proliferative,3,endometrium,9863,96357,4496
6,UA_Endo10298210_FX1176,UA_Endo10298210,FX1176,2535,36601,9908912,float32,csr,Mareckova-cells,case,natural_cycle,Proliferative,4,endometrium,9863,96357,4496
7,UA_Endo10298211_FX1125,UA_Endo10298211,FX1125,45,36601,269696,float32,csr,Mareckova-cells & nuclei,case,natural_cycle,Proliferative,3,endometrium,2247,307686,4919
8,UA_Endo10298211_FX1176,UA_Endo10298211,FX1176,943,36601,5948980,float32,csr,Mareckova-cells,case,natural_cycle,Proliferative,4,endometrium,2247,307686,4919
9,UA_Endo10298212_FX1156,UA_Endo10298212,FX1156,3229,36601,8951641,float32,csr,Mareckova-cells & nuclei,case,natural_cycle,Secretory Mid,4,endometrium,19057,42932,2834


### Section 1 checkpoint

At the end of this section, `sample_adatas` contains one raw-count `AnnData` object per donor–library archive, every object carries validated biological metadata, and `sample_summary` describes both the matrix collection and its study design. No expression values have been changed.

The next section will concatenate the objects safely, calculate per-cell and per-gene QC metrics, apply paper-aligned quality filters, preserve raw counts, and normalize expression values.